# 04 — Clasificador de tipo de ataque (XGBoost)

Segunda etapa del pipeline. Toma las muestras y clasifica entre `Normal`, `DDoS`, `DoS`, `Probe`, `BFA`.

- Mismas 7 features deployable (las que la Ryu app va a producir en vivo).
- Se entrena con todas las clases para que pueda **rechazar falsos positivos** del detector de anomalías diciendo `Normal`.
- Descartamos `U2R` (solo 17 muestras → no es viable clasificarla).

Al final componemos el pipeline completo: AE (anomalía) → XGBoost (tipo).

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, str(Path.cwd().parent))
from src.data import get_insdn_path
from src.features import clean_insdn, INSDN_TO_OPENFLOW

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
MODELS = Path('../models')

## 1. Carga y preparación

In [ ]:
csv = next(get_insdn_path().rglob('*.csv'))
df = pd.read_csv(csv, low_memory=False)
df.columns = df.columns.str.strip()
df = clean_insdn(df)

# Descartar U2R (solo 17 muestras)
before = len(df)
df = df[df['Label'] != 'U2R'].reset_index(drop=True)
print(f'Eliminadas {before - len(df)} muestras de U2R; quedan {len(df)} filas.')

X = df[list(INSDN_TO_OPENFLOW.keys())].copy()
X['Flow Duration'] = X['Flow Duration'] / 1e6
X = X.rename(columns=INSDN_TO_OPENFLOW)
y_str = df['Label'].values

le = LabelEncoder().fit(y_str)
y = le.transform(y_str)
print('Clases:', dict(zip(le.classes_, np.bincount(y))))

## 2. Split estratificado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 3. Entrenamiento XGBoost

Pesos de muestra inversamente proporcionales a la frecuencia de la clase: así DoS y BFA (minoritarias) no se ignoran.

In [ ]:
class_counts = np.bincount(y_train)
class_weight = len(y_train) / (len(class_counts) * class_counts)
sample_weight = class_weight[y_train]
for c, n, w in zip(le.classes_, class_counts, class_weight):
    print(f'  {c:8s}  n={n:>6d}  weight={w:.3f}')

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=len(le.classes_),
    tree_method='hist',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
)
clf.fit(X_train, y_train, sample_weight=sample_weight)
print('Entrenado.')

In [ ]:
y_pred = clf.predict(X_test)
print(f'F1 macro:    {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'F1 weighted: {f1_score(y_test, y_pred, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
)
ax.set_title('XGBoost — matriz de confusión normalizada (por fila)')
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout()

## 4. Importancia de features

In [ ]:
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
imp.plot(kind='barh', ax=ax)
ax.set_title('Importancia de features (gain)')
plt.tight_layout()

## 5. Pipeline completo: AE (anomalía) → XGBoost (tipo)

Simulamos el flujo real:

1. Para cada flow, el Autoencoder calcula error de reconstrucción.
2. Si supera el umbral → pasa al clasificador para etiquetar el tipo.
3. Si no → se etiqueta directamente `Normal` (no se pasa al clasificador para ahorrar latencia).

Requisito: haber ejecutado antes `03_anomaly_detector.ipynb` para que existan `scaler.pkl`, `autoencoder.pt` y `detector_meta.pkl` en `models/`.

In [ ]:
scaler = joblib.load(MODELS / 'scaler.pkl')
meta = joblib.load(MODELS / 'detector_meta.pkl')
ae_thr = meta['ae_thr_p99']

class Autoencoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 4),
        )
        self.decoder = nn.Sequential(
            nn.Linear(4, 8), nn.ReLU(),
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, in_dim),
        )
    def forward(self, x): return self.decoder(self.encoder(x))

ae = Autoencoder(in_dim=meta['ae_arch']['in_dim'])
ae.load_state_dict(torch.load(MODELS / 'autoencoder.pt', map_location='cpu'))
ae.eval()
print(f'AE cargado, threshold P99 = {ae_thr:.5f}')

In [ ]:
X_test_s = scaler.transform(X_test)
with torch.no_grad():
    xt = torch.tensor(X_test_s, dtype=torch.float32)
    ae_err = ((ae(xt) - xt) ** 2).mean(dim=1).numpy()

normal_idx = le.transform(['Normal'])[0]
y_pipeline = np.full(len(X_test), normal_idx, dtype=int)
anomalous = ae_err > ae_thr
print(f'Flagged como anómalos por AE: {anomalous.sum()} / {len(X_test)}')
if anomalous.sum() > 0:
    y_pipeline[anomalous] = clf.predict(X_test.iloc[anomalous])

print(f'\n=== Pipeline AE+XGB (umbral fijo P99) ===')
print(f'F1 macro:    {f1_score(y_test, y_pipeline, average="macro"):.4f}')
print(f'F1 weighted: {f1_score(y_test, y_pipeline, average="weighted"):.4f}')
print(classification_report(y_test, y_pipeline, target_names=le.classes_, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pipeline, normalize='true')
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
)
ax.set_title('Pipeline AE+XGB — matriz de confusión')
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout()

## 6. Persistencia

In [ ]:
joblib.dump(clf, MODELS / 'xgb_classifier.pkl')
joblib.dump(le, MODELS / 'label_encoder.pkl')
print('Guardados xgb_classifier.pkl y label_encoder.pkl en', MODELS.resolve())

## Próximos pasos

- Implementar `src/detector.py` cargando los 4 artefactos (`scaler`, `autoencoder`, `xgb_classifier`, `label_encoder`) y exponiendo `predict()`.
- Coordinar con subgrupo B la integración en la Ryu app y la validación en Mininet.